### Installing Libraries

In [ ]:
!pip install ultralytics -q

### Verifying Each Dataset

In [ ]:
from pathlib import Path
from collections import Counter


def verify_yolo_dataset(dataset_dir):
    dataset_dir = Path(dataset_dir)

    if not dataset_dir.exists():
        print(f"Error: Directory '{dataset_dir}' does not exist.")
        return

    splits = ["train", "valid", "val", "test"]
    total_images = 0
    total_instances = 0
    overall_class_counts = Counter()

    print("=" * 50)
    print(" YOLO DATASET SUMMARY")
    print("=" * 50)

    for split in splits:
        split_dir = dataset_dir / split
        if not split_dir.exists():
            continue

        # Look for images in split/images or directly in split/
        img_dir = split_dir / "images" if (split_dir / "images").exists() else split_dir
        lbl_dir = split_dir / "labels" if (split_dir / "labels").exists() else split_dir

        images = list(img_dir.rglob("*.jpg"))
        split_class_counts = Counter()

        for img_path in images:
            # Find matching label file (.txt)
            lbl_path = lbl_dir / f"{img_path.stem}.txt"
            if lbl_path.exists():
                with open(lbl_path, "r") as f:
                    for line in f:
                        parts = line.strip().split()
                        if parts:
                            class_id = int(float(parts[0]))
                            split_class_counts[class_id] += 1

        img_count = len(images)
        inst_count = sum(split_class_counts.values())

        total_images += img_count
        total_instances += inst_count
        overall_class_counts.update(split_class_counts)

        # Print Split Results
        print(f"\n[ Split: '{split}' ]")
        print(f"  Images: {img_count} | Instances: {inst_count}")
        for cid, count in sorted(split_class_counts.items()):
            print(f"    Class {cid}: {count}")

    # Print Totals
    print("\n" + "=" * 50)
    print(f"Total Images    : {total_images}")
    print(f"Total Instances : {total_instances}")
    print("Overall Class Totals:")
    for cid, count in sorted(overall_class_counts.items()):
        print(f"  Class {cid}: {count}")
    print("=" * 50)


In [ ]:
if __name__ == "__main__":
    verify_yolo_dataset("Whitefly-Adults-3")
    verify_yolo_dataset("Whitefly-Nymph-3")

### Creating Patches

In [ ]:
import os
import shutil
from pathlib import Path
from typing import List, Tuple, Set
import cv2
import numpy as np

class YOLOPatchGenerator:
    def __init__(self, source_dir: str, target_dir: str, patch_size: int = 224, original_size: int = 1024, min_iou: float = 0.7):
        """
        Initialize the patch generator.
        
        Args:
            source_dir: Source dataset directory
            target_dir: Target dataset directory
            patch_size: Size of the patches to generate (default: 224)
            original_size: Size of original images (default: 1024)
            min_iou: Minimum IoU for a box to be considered "covered" (default: 0.7)
        """
        self.source_dir = Path(source_dir)
        self.target_dir = Path(target_dir)
        self.patch_size = patch_size
        self.original_size = original_size
        self.min_iou = min_iou
        
    def parse_yolo_label(self, label_path: str, img_width: int, img_height: int) -> List[Tuple]:
        """Parse YOLO format label file and return bounding boxes."""
        boxes = []
        if not os.path.exists(label_path):
            return boxes
            
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_id = int(parts[0])
                    x_center = float(parts[1]) * img_width
                    y_center = float(parts[2]) * img_height
                    width = float(parts[3]) * img_width
                    height = float(parts[4]) * img_height
                    
                    # Convert to corner coordinates
                    x1 = x_center - width / 2
                    y1 = y_center - height / 2
                    x2 = x_center + width / 2
                    y2 = y_center + height / 2
                    
                    boxes.append((class_id, x1, y1, x2, y2))
        return boxes
    
    def calculate_iou(self, box1: Tuple[float, float, float, float], 
                     box2: Tuple[float, float, float, float]) -> float:
        """Calculate IoU between two boxes (x1, y1, x2, y2)."""
        x1_inter = max(box1[0], box2[0])
        y1_inter = max(box1[1], box2[1])
        x2_inter = min(box1[2], box2[2])
        y2_inter = min(box1[3], box2[3])
        
        if x2_inter < x1_inter or y2_inter < y1_inter:
            return 0.0
        
        inter_area = (x2_inter - x1_inter) * (y2_inter - y1_inter)
        box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
        box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
        union_area = box1_area + box2_area - inter_area
        
        return inter_area / union_area if union_area > 0 else 0.0
    
    def box_in_patch(self, box: Tuple, patch_x: int, patch_y: int) -> bool:
        """Check if a bounding box intersects with a patch."""
        _, x1, y1, x2, y2 = box
        patch_x2 = patch_x + self.patch_size
        patch_y2 = patch_y + self.patch_size
        
        # Check if there's any overlap
        return not (x2 < patch_x or x1 > patch_x2 or y2 < patch_y or y1 > patch_y2)
    
    def get_box_coverage_in_patch(self, box: Tuple, patch_x: int, patch_y: int) -> float:
        """Calculate what fraction of the box is covered by the patch."""
        _, x1, y1, x2, y2 = box
        patch_x2 = patch_x + self.patch_size
        patch_y2 = patch_y + self.patch_size
        
        # Calculate intersection
        x1_inter = max(x1, patch_x)
        y1_inter = max(y1, patch_y)
        x2_inter = min(x2, patch_x2)
        y2_inter = min(y2, patch_y2)
        
        if x2_inter <= x1_inter or y2_inter <= y1_inter:
            return 0.0
        
        inter_area = (x2_inter - x1_inter) * (y2_inter - y1_inter)
        box_area = (x2 - x1) * (y2 - y1)
        
        return inter_area / box_area if box_area > 0 else 0.0
    
    def find_optimal_patch_positions(self, boxes: List[Tuple], img_width: int, img_height: int) -> List[Tuple[int, int]]:
        """Find minimal set of patch positions that cover all boxes well."""
        if not boxes:
            # No boxes - return center patch
            center_x = (img_width - self.patch_size) // 2
            center_y = (img_height - self.patch_size) // 2
            return [(center_x, center_y)]
        
        # Start with patches centered on each box
        patch_positions = []
        covered_boxes = set()
        
        for idx, box in enumerate(boxes):
            _, x1, y1, x2, y2 = box
            box_center_x = (x1 + x2) / 2
            box_center_y = (y1 + y2) / 2
            
            # Position patch centered on box
            patch_x = int(box_center_x - self.patch_size / 2)
            patch_y = int(box_center_y - self.patch_size / 2)
            
            # Ensure patch is within image bounds
            patch_x = max(0, min(patch_x, img_width - self.patch_size))
            patch_y = max(0, min(patch_y, img_height - self.patch_size))
            
            # Check coverage
            coverage = self.get_box_coverage_in_patch(box, patch_x, patch_y)
            
            if coverage >= self.min_iou:
                # This patch covers the current box well
                patch_positions.append((patch_x, patch_y))
                covered_boxes.add(idx)
                
                # Check if this patch also covers other boxes well
                for other_idx, other_box in enumerate(boxes):
                    if other_idx != idx and other_idx not in covered_boxes:
                        other_coverage = self.get_box_coverage_in_patch(other_box, patch_x, patch_y)
                        if other_coverage >= self.min_iou:
                            covered_boxes.add(other_idx)
        
        # Check for any uncovered boxes
        uncovered = set(range(len(boxes))) - covered_boxes
        
        # For uncovered boxes, try to find patches that cover multiple boxes
        for idx in uncovered:
            box = boxes[idx]
            _, x1, y1, x2, y2 = box
            
            # Try multiple positions around the box
            best_position = None
            best_score = 0
            
            positions_to_try = [
                # Centered on box
                (int((x1 + x2) / 2 - self.patch_size / 2), 
                 int((y1 + y2) / 2 - self.patch_size / 2)),
                # Aligned to box corners
                (int(x1), int(y1)),
                (int(x2 - self.patch_size), int(y1)),
                (int(x1), int(y2 - self.patch_size)),
                (int(x2 - self.patch_size), int(y2 - self.patch_size)),
            ]
            
            for patch_x, patch_y in positions_to_try:
                # Ensure within bounds
                patch_x = max(0, min(patch_x, img_width - self.patch_size))
                patch_y = max(0, min(patch_y, img_height - self.patch_size))
                
                # Score based on how many uncovered boxes this would cover
                score = 0
                for uncov_idx in uncovered:
                    coverage = self.get_box_coverage_in_patch(boxes[uncov_idx], patch_x, patch_y)
                    if coverage >= self.min_iou:
                        score += 1
                
                if score > best_score:
                    best_score = score
                    best_position = (patch_x, patch_y)
            
            if best_position and best_position not in patch_positions:
                patch_positions.append(best_position)
        
        # Remove duplicate positions (within small tolerance)
        unique_positions = []
        tolerance = 20  # pixels
        
        for pos in patch_positions:
            is_duplicate = False
            for unique_pos in unique_positions:
                if abs(pos[0] - unique_pos[0]) < tolerance and abs(pos[1] - unique_pos[1]) < tolerance:
                    is_duplicate = True
                    break
            if not is_duplicate:
                unique_positions.append(pos)
        
        return unique_positions
    
    def convert_box_to_patch_coords(self, box: Tuple, patch_x: int, patch_y: int) -> Tuple:
        """Convert bounding box coordinates to patch-relative YOLO format with full range clamping."""
        class_id, x1, y1, x2, y2 = box

        # Clip to patch boundaries (raw coords)
        x1_clipped = max(0.0, x1 - patch_x)
        y1_clipped = max(0.0, y1 - patch_y)
        x2_clipped = min(float(self.patch_size), x2 - patch_x)
        y2_clipped = min(float(self.patch_size), y2 - patch_y)

        # Manual correction for any tiny floating errors at these raw coordinates
        if x1_clipped < 0.0:
            x1_clipped = 0.0
        if y1_clipped < 0.0:
            y1_clipped = 0.0
        if x2_clipped > float(self.patch_size):
            x2_clipped = float(self.patch_size)
        if y2_clipped > float(self.patch_size):
            y2_clipped = float(self.patch_size)

        # Ensure valid box (min < max)
        if x2_clipped <= x1_clipped or y2_clipped <= y1_clipped:
            return None

        # Convert to YOLO format (normalized center x, center y, width, height)
        width = x2_clipped - x1_clipped
        height = y2_clipped - y1_clipped
        x_center = (x1_clipped + x2_clipped) / 2.0 / self.patch_size
        y_center = (y1_clipped + y2_clipped) / 2.0 / self.patch_size
        width_norm = width / self.patch_size
        height_norm = height / self.patch_size

        # Forcefully clamp normalized values to [0.0, 1.0] using explicit checks
        if x_center < 0.0:
            x_center = 0.0
        elif x_center > 1.0:
            x_center = 1.0

        if y_center < 0.0:
            y_center = 0.0
        elif y_center > 1.0:
            y_center = 1.0

        if width_norm < 0.0:
            width_norm = 0.0
        elif width_norm > 1.0:
            width_norm = 1.0

        if height_norm < 0.0:
            height_norm = 0.0
        elif height_norm > 1.0:
            height_norm = 1.0

        # Skip boxes that collapse to zero after normalization
        if width_norm <= 0.0 or height_norm <= 0.0:
            return None

        return (class_id, x_center, y_center, width_norm, height_norm)
    
    def generate_patches_for_image(self, img_path: str, label_path: str, 
                                   output_img_dir: str, output_label_dir: str,
                                   img_name: str):
        """Generate patches for a single image."""
        # Read image
        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Could not read image {img_path}")
            return
        
        img_height, img_width = img.shape[:2]
        
        # Parse labels
        boxes = self.parse_yolo_label(label_path, img_width, img_height)
        
        # Find optimal patch positions
        patch_positions = self.find_optimal_patch_positions(boxes, img_width, img_height)
        
        # Create patches
        for patch_idx, (patch_x, patch_y) in enumerate(patch_positions):
            self.create_patch(img, boxes, patch_x, patch_y,
                            output_img_dir, output_label_dir,
                            img_name, patch_idx)
        
        return len(patch_positions)
    
    def create_patch(self, img, boxes, patch_x, patch_y,
                    output_img_dir, output_label_dir, img_name, patch_count):
        """Create a single patch and its label file."""
        patch = img[patch_y:patch_y+self.patch_size, patch_x:patch_x+self.patch_size]
        
        # Find boxes in this patch
        patch_boxes = []
        for box in boxes:
            if self.box_in_patch(box, patch_x, patch_y):
                converted_box = self.convert_box_to_patch_coords(box, patch_x, patch_y)
                if converted_box:  # <- skip invalid boxes
                    patch_boxes.append(converted_box)
        
        # Save patch image
        patch_name = f"{img_name}_patch_{patch_count}"
        cv2.imwrite(os.path.join(output_img_dir, f"{patch_name}.jpg"), patch)
        
        # Save patch labels
        with open(os.path.join(output_label_dir, f"{patch_name}.txt"), 'w') as f:
            for box in patch_boxes:
                class_id, x_center, y_center, width, height = box
                f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")
    
    def process_split(self, split: str):
        """Process a dataset split (train/val/test)."""
        print(f"Processing {split} split...")
        
        source_img_dir = self.source_dir / split / "images"
        source_label_dir = self.source_dir / split / "labels"
        
        target_img_dir = self.target_dir / split / "images"
        target_label_dir = self.target_dir / split / "labels"
        
        # Create output directories
        target_img_dir.mkdir(parents=True, exist_ok=True)
        target_label_dir.mkdir(parents=True, exist_ok=True)
        
        # Process each image
        if not source_img_dir.exists():
            print(f"Warning: {source_img_dir} does not exist, skipping...")
            return
        
        image_files = list(source_img_dir.glob("*.jpg")) + list(source_img_dir.glob("*.png"))
        
        total_patches = 0
        for img_path in image_files:
            img_name = img_path.stem
            label_path = source_label_dir / f"{img_name}.txt"
            
            num_patches = self.generate_patches_for_image(
                str(img_path), 
                str(label_path),
                str(target_img_dir),
                str(target_label_dir),
                img_name
            )
            total_patches += num_patches
            print(f"  {img_name}: {num_patches} patches")
        
        avg_patches = total_patches / len(image_files) if image_files else 0
        print(f"  Total: {total_patches} patches from {len(image_files)} images (avg: {avg_patches:.1f} per image)\n")
    
    def generate_dataset(self):
        """Generate the complete patched dataset."""
        print(f"Generating patches from {self.source_dir} to {self.target_dir}")
        print(f"Patch size: {self.patch_size}x{self.patch_size}")
        print(f"Minimum box coverage (IoU): {self.min_iou:.2f}")
        print()
        
        # Process each split
        for split in ['train', 'valid', 'test']:
            self.process_split(split)
        
        print("\nDataset generation complete!")


if __name__ == "__main__":
    # Initialize and run the patch generator
    adult_generator = YOLOPatchGenerator(
        source_dir="Whitefly-Adults-3",
        target_dir="Whitefly-Adults-Patches",
        patch_size=224,
        original_size=1024,
        min_iou=0.8
    )

    nymph_generator = YOLOPatchGenerator(
        source_dir="Whitefly-Nymph-3",
        target_dir="Whitefly-Nymph-Patches",
        patch_size=224,
        original_size=1024,
        min_iou=0.8
    )
    
    adult_generator.generate_dataset()
    nymph_generator.generate_dataset()


### Rebalancing Datasets

In [ ]:
import os
import shutil
from pathlib import Path
import random

class DatasetRebalancer:
    def __init__(self, dataset_dir: str, train_ratio: float = 0.7, 
                 valid_ratio: float = 0.2, test_ratio: float = 0.1):
        """
        Initialize the dataset rebalancer.
        
        Args:
            dataset_dir: Path to the dataset directory
            train_ratio: Proportion for training set (default: 0.7)
            valid_ratio: Proportion for validation set (default: 0.2)
            test_ratio: Proportion for test set (default: 0.1)
        """
        self.dataset_dir = Path(dataset_dir)
        self.train_ratio = train_ratio
        self.valid_ratio = valid_ratio
        self.test_ratio = test_ratio
        
        # Verify ratios sum to 1
        total = train_ratio + valid_ratio + test_ratio
        if abs(total - 1.0) > 0.01:
            raise ValueError(f"Ratios must sum to 1.0, got {total}")
    
    def rebalance(self, seed: int = 42):
        """Rebalance the dataset splits."""
        random.seed(seed)
        
        print("="*60)
        print("DATASET REBALANCER")
        print("="*60)
        print(f"Dataset: {self.dataset_dir}")
        print(f"Target split: Train={self.train_ratio:.1%}, Valid={self.valid_ratio:.1%}, Test={self.test_ratio:.1%}")
        print()
        
        # Create temp directory and collect all files
        temp_dir = self.dataset_dir / "temp_all_files"
        temp_images = temp_dir / "images"
        temp_labels = temp_dir / "labels"
        
        print("Step 1: Collecting all files into temp folder...")
        temp_images.mkdir(parents=True, exist_ok=True)
        temp_labels.mkdir(parents=True, exist_ok=True)
        
        # Collect from all splits
        all_image_files = []
        for split in ['train', 'valid', 'test']:
            img_dir = self.dataset_dir / split / "images"
            label_dir = self.dataset_dir / split / "labels"
            
            if img_dir.exists():
                for img_file in img_dir.glob("*.*"):
                    dest = temp_images / img_file.name
                    shutil.copy2(img_file, dest)
                    all_image_files.append(img_file.name)
            
            if label_dir.exists():
                for label_file in label_dir.glob("*.txt"):
                    dest = temp_labels / label_file.name
                    shutil.copy2(label_file, dest)
        
        print(f"Collected {len(all_image_files)} image files")
        
        # Shuffle and split
        print("\nStep 2: Shuffling and splitting files...")
        random.shuffle(all_image_files)
        
        total = len(all_image_files)
        train_count = int(total * self.train_ratio)
        valid_count = int(total * self.valid_ratio)
        
        train_files = all_image_files[:train_count]
        valid_files = all_image_files[train_count:train_count + valid_count]
        test_files = all_image_files[train_count + valid_count:]
        
        print(f"  Train: {len(train_files)} files ({len(train_files)/total*100:.1f}%)")
        print(f"  Valid: {len(valid_files)} files ({len(valid_files)/total*100:.1f}%)")
        print(f"  Test: {len(test_files)} files ({len(test_files)/total*100:.1f}%)")
        
        # Clear existing splits
        print("\nStep 3: Clearing existing split directories...")
        for split in ['train', 'valid', 'test']:
            split_dir = self.dataset_dir / split
            if split_dir.exists():
                shutil.rmtree(split_dir)
            (split_dir / "images").mkdir(parents=True, exist_ok=True)
            (split_dir / "labels").mkdir(parents=True, exist_ok=True)
        
        # Distribute files to splits
        print("\nStep 4: Distributing files to new splits...")
        
        splits = {
            'train': train_files,
            'valid': valid_files,
            'test': test_files
        }
        
        for split_name, file_list in splits.items():
            dest_img_dir = self.dataset_dir / split_name / "images"
            dest_label_dir = self.dataset_dir / split_name / "labels"
            
            for img_name in file_list:
                # Copy image
                src_img = temp_images / img_name
                dest_img = dest_img_dir / img_name
                shutil.copy2(src_img, dest_img)
                
                # Copy corresponding label
                label_name = Path(img_name).stem + ".txt"
                src_label = temp_labels / label_name
                dest_label = dest_label_dir / label_name
                if src_label.exists():
                    shutil.copy2(src_label, dest_label)
            
            print(f"  {split_name}: {len(file_list)} files copied")
        
        # Clean up temp directory
        print("\nStep 5: Cleaning up temp directory...")
        shutil.rmtree(temp_dir)
        
        print("\n" + "="*60)
        print("REBALANCING COMPLETE!")
        print("="*60)
        print(f"\nFinal distribution:")
        print(f"  Train: {len(train_files)} files ({len(train_files)/total*100:.1f}%)")
        print(f"  Valid: {len(valid_files)} files ({len(valid_files)/total*100:.1f}%)")
        print(f"  Test: {len(test_files)} files ({len(test_files)/total*100:.1f}%)")
        print()


if __name__ == "__main__":
    # Initialize and run the rebalancer
    adult_rebalancer = DatasetRebalancer(
        dataset_dir="Whitefly-Adults-Patches",
        train_ratio=0.7,
        valid_ratio=0.2,
        test_ratio=0.1
    )

    nymph_rebalancer = DatasetRebalancer(
        dataset_dir="Whitefly-Nymph-Patches",
        train_ratio=0.7,
        valid_ratio=0.2,
        test_ratio=0.1
    )
    
    adult_rebalancer.rebalance(seed=42)
    nymph_rebalancer.rebalance(seed=42)

### Apply Augmentation

In [ ]:
import os
import cv2
from pathlib import Path
import albumentations as A
from tqdm import tqdm

class YOLOAugmentor:
    def __init__(self, dataset_path, augmentation_factor=5):
        """
        Initialize augmentor for YOLO format dataset (v8, v9, v10, v11)
        
        Args:
            dataset_path: Path to folder containing 'train', 'valid', and 'test' folders
            augmentation_factor: How many augmented copies to create per image
        """
        self.dataset_path = Path(dataset_path)
        self.augmentation_factor = augmentation_factor
        
        # Define augmentation pipeline
        self.transform = A.Compose([
            # Geometric transformations
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Rotate(limit=15, p=0.5, border_mode=cv2.BORDER_CONSTANT),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.3),
            
            # Color augmentations
            A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.03, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.4),
            A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=10, val_shift_limit=10, p=0.4),
            
            A.RandomToneCurve(scale=0.1, p=0.3),

            # Blur
            A.OneOf([
                A.Blur(blur_limit=3, p=1.0),
                A.GaussianBlur(blur_limit=3, p=1.0),
                A.MotionBlur(blur_limit=3, p=1.0),
            ], p=0.3),
            
            # Sharpening
            A.Sharpen(alpha=(0.1, 0.3), lightness=(0.8, 1.0), p=0.3),
            
            # Additional augmentations
            A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.2),
            A.RandomGamma(gamma_limit=(90, 110), p=0.15),
            
            # Subtle lens distortions
            A.OpticalDistortion(distort_limit=0.05, p=0.3),
            
        ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.3))
    
    def load_yolo_annotations(self, label_path):
        """Load YOLO format annotations from text file"""
        bboxes = []
        class_labels = []
        
        if not os.path.exists(label_path):
            return bboxes, class_labels
        
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_id = int(parts[0])
                    x_center, y_center, width, height = map(float, parts[1:5])
                    bboxes.append([x_center, y_center, width, height])
                    class_labels.append(class_id)
        
        return bboxes, class_labels
    
    def save_yolo_annotations(self, label_path, bboxes, class_labels):
        """Save YOLO format annotations to text file"""
        with open(label_path, 'w') as f:
            for bbox, class_id in zip(bboxes, class_labels):
                line = f"{class_id} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n"
                f.write(line)
        
    def augment_image(self, image, bboxes, class_labels):
        """Apply augmentation to image and bounding boxes"""
        if len(bboxes) == 0:
            # If no bboxes, apply image-only augmentation without bbox_params
            transform_no_bbox = A.Compose([
                # Geometric transformations
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.Rotate(limit=15, p=0.5, border_mode=cv2.BORDER_CONSTANT),
                A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.3),
                
                # Color augmentations
                A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.03, p=0.5),
                A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.4),
                A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=10, val_shift_limit=10, p=0.4),
                A.RandomToneCurve(scale=0.1, p=0.3),
    
                # Blur
                A.OneOf([
                    A.Blur(blur_limit=3, p=1.0),
                    A.GaussianBlur(blur_limit=3, p=1.0),
                    A.MotionBlur(blur_limit=3, p=1.0),
                ], p=0.3),
                
                # Sharpening
                A.Sharpen(alpha=(0.1, 0.3), lightness=(0.8, 1.0), p=0.3),
                
                # Additional augmentations
                A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.2),
                A.RandomGamma(gamma_limit=(90, 110), p=0.15),
                A.OpticalDistortion(distort_limit=0.05, p=0.3),
            ])
            augmented = transform_no_bbox(image=image)
            return augmented['image'], [], []
        
        try:
            augmented = self.transform(image=image, bboxes=bboxes, class_labels=class_labels)
            return augmented['image'], augmented['bboxes'], augmented['class_labels']
        except Exception as e:
            print(f"Augmentation error: {e}")
            return image, bboxes, class_labels
        
    def process_split(self, split_name):
        """Process a dataset split (train, valid, or test)"""
        split_path = self.dataset_path / split_name
        images_path = split_path / 'images'
        labels_path = split_path / 'labels'
        
        if not images_path.exists():
            print(f"Skipping {split_name}: images folder not found")
            return
        
        # Get all image files
        image_files = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
            image_files.extend(list(images_path.glob(ext)))
        
        print(f"\nProcessing {split_name} split: {len(image_files)} images")
        print(f"Creating {self.augmentation_factor} augmented versions per image")
        
        # Process each image
        for img_path in tqdm(image_files, desc=f"Augmenting {split_name}"):
            # Load image
            image = cv2.imread(str(img_path))
            if image is None:
                print(f"Failed to load: {img_path}")
                continue
            
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            # Load annotations
            label_path = labels_path / (img_path.stem + '.txt')
            bboxes, class_labels = self.load_yolo_annotations(label_path)
            
            # Create augmented versions
            for aug_idx in range(self.augmentation_factor):
                # Apply augmentation
                aug_image, aug_bboxes, aug_class_labels = self.augment_image(
                    image.copy(), bboxes.copy(), class_labels.copy()
                )
                
                # Save augmented image
                aug_image_bgr = cv2.cvtColor(aug_image, cv2.COLOR_RGB2BGR)
                aug_img_name = f"{img_path.stem}_aug{aug_idx}{img_path.suffix}"
                aug_img_path = images_path / aug_img_name
                cv2.imwrite(str(aug_img_path), aug_image_bgr)
                
                # Save augmented annotations
                aug_label_path = labels_path / (img_path.stem + f'_aug{aug_idx}.txt')
                self.save_yolo_annotations(aug_label_path, aug_bboxes, aug_class_labels)
        
        # Count final images
        final_count = len(list(images_path.glob('*')))
        print(f"✓ {split_name} complete: {final_count} total images (original + augmented)")
    
    def run(self):
        """Run augmentation on train split"""
        print("=" * 60)
        print("YOLO Dataset Augmentation (v8, v9, v10, v11)")
        print("=" * 60)
        print(f"Dataset path: {self.dataset_path}")
        print(f"Augmentation factor: {self.augmentation_factor}x")
        print(f"Expected final size: ~{self.augmentation_factor + 1}x original")
        
        # Process train split
        self.process_split('train')
        
        print("\n" + "=" * 60)
        print("Augmentation complete!")
        print("=" * 60)


if __name__ == "__main__":
    # Configure and run the augmentor
    adult_augmentor = YOLOAugmentor(
        "Whitefly-Adults-Patches", 
        augmentation_factor=4
    )

    nymph_augmentor = YOLOAugmentor(
        "Whitefly-Nymph-Patches",
        augmentation_factor=8
    )
    
    adult_augmentor.run()
    nymph_augmentor.run()

### Combining Datasets

In [ ]:
import os
import shutil
from pathlib import Path
import yaml

def relabel_yolo_annotations(label_file, old_class=0, new_class=1):
    """
    Read YOLO annotation file and change class ID from old_class to new_class
    """
    with open(label_file, 'r') as f:
        lines = f.readlines()
    
    new_lines = []
    for line in lines:
        parts = line.strip().split()
        if parts:
            # Convert to float first, then to int to handle both "0" and "0.0"
            current_class = int(float(parts[0]))
            if current_class == old_class:
                parts[0] = str(new_class)
            else:
                # Ensure class ID is integer format even if not relabeling
                parts[0] = str(current_class)
        new_lines.append(' '.join(parts) + '\n')
    
    return new_lines

def copy_dataset_split(src_folder, dest_folder, split_name, relabel=False):
    """
    Copy images and labels from source split to destination split
    Args:
        src_folder: Source dataset folder
        dest_folder: Destination dataset folder
        split_name: 'train', 'valid', or 'test'
        relabel: If True, change class 0 to class 1 in labels
    """
    src_images = Path(src_folder) / split_name / 'images'
    src_labels = Path(src_folder) / split_name / 'labels'
    
    dest_images = Path(dest_folder) / split_name / 'images'
    dest_labels = Path(dest_folder) / split_name / 'labels'
    
    # Create destination directories
    dest_images.mkdir(parents=True, exist_ok=True)
    dest_labels.mkdir(parents=True, exist_ok=True)
    
    # Copy images
    if src_images.exists():
        for img_file in src_images.glob('*'):
            if img_file.is_file():
                shutil.copy2(img_file, dest_images / img_file.name)
                # print(f"Copied image: {img_file.name}")
    
    # Copy and relabel annotations if needed
    if src_labels.exists():
        for label_file in src_labels.glob('*.txt'):
            if relabel:
                # Relabel from class 0 to class 1
                new_lines = relabel_yolo_annotations(label_file, old_class=0, new_class=1)
                with open(dest_labels / label_file.name, 'w') as f:
                    f.writelines(new_lines)
                # print(f"Relabeled and copied: {label_file.name}")
            else:
                # Just copy as is
                shutil.copy2(label_file, dest_labels / label_file.name)
                # print(f"Copied label: {label_file.name}")

def create_data_yaml(dest_folder):
    """
    Create data.yaml file for the combined dataset
    """
    data_yaml_content = {
        'train': '../train/images',
        'val': '../valid/images',
        'test': '../test/images',
        'nc': 2,
        'names': ['adult', 'nymph']
    }
    
    yaml_path = Path(dest_folder) / 'data.yaml'
    with open(yaml_path, 'w') as f:
        yaml.dump(data_yaml_content, f, default_flow_style=False, sort_keys=False)
    
    print(f"\nCreated data.yaml at {yaml_path}")

def combine_whitefly_datasets(adults_folder, nymph_folder, output_folder):
    """
    Main function to combine two YOLO datasets
    Args:
        adults_folder: Path to adults dataset folder
        nymph_folder: Path to nymphs dataset folder
        output_folder: Path to new multi stage folder
    """
    print("Starting dataset combination...")
    print(f"Adults dataset: {adults_folder}")
    print(f"Nymph dataset: {nymph_folder}")
    print(f"Output dataset: {output_folder}\n")
    
    # Create output folder
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    # Process each split (train, valid, test)
    for split in ['train', 'valid', 'test']:
        print(f"\n{'='*50}")
        print(f"Processing {split} split...")
        print(f"{'='*50}")
        
        # Copy adults dataset
        print(f"\nCopying adults {split} data")
        copy_dataset_split(adults_folder, output_folder, split, relabel=False)
        
        # Copy nymph dataset
        print(f"\nCopying nymph {split} data")
        copy_dataset_split(nymph_folder, output_folder, split, relabel=False)
    
    # Create data.yaml file
    print(f"\n{'='*50}")
    create_data_yaml(output_folder)
    print(f"{'='*50}")
    
    print("\n✓ Dataset combination complete!")
    print(f"Combined dataset created at: {output_folder}")

# Usage
if __name__ == "__main__":
    # Define paths
    ADULTS_FOLDER = "Whitefly-Adults-Patches"
    NYMPH_FOLDER = "Whitefly-Nymph-Patches"
    OUTPUT_FOLDER = "Whitefly-Multi-Stage"
    
    # Run the combination
    combine_whitefly_datasets(ADULTS_FOLDER, NYMPH_FOLDER, OUTPUT_FOLDER)

### Verifying Combined Dataset

In [ ]:
if __name__ == "__main__":
    verify_yolo_dataset("Whitefly-Multi-Stage")

### Training Model (Backbone and Neck Frozen)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8x.pt")

model.train(
    data="Whitefly-Multi-Stage/data.yaml",
    epochs=20,
    imgsz=224,
    device=[0, 1],
    batch=16,
    freeze=21,
    seed=42
)

### Training Model (Backbone and Neck Unfrozen)

In [ ]:
from ultralytics import YOLO

model = YOLO("runs/detect/train/weights/last.pt")

model.train(
    data="Whitefly-Multi-Stage/data.yaml",
    epochs=100,
    imgsz=224,
    device=[0, 1],
    batch=16,
    lr0=0.001,
    lrf=0.0001,
    freeze=0,
    seed=42
)

### Validation Metrics

In [ ]:
# Load best model
model = YOLO("runs/detect/train-2/weights/best.pt")

# Validate the model
metrics = model.val()

print()
print("map50-95 \t", metrics.box.map)    # map50-95
print("map50    \t", metrics.box.map50)  # map50

print("Precision\t", metrics.box.p)      # overall precision
print("Recall   \t", metrics.box.r)      # overall recall

### Testing Metrics

In [ ]:
# Load best model
model = YOLO("runs/detect/train-2/weights/best.pt")

# Validate the model
metrics = model.val(split='test')

print()
print("map50-95 \t", metrics.box.map)    # map50-95
print("map50    \t", metrics.box.map50)  # map50

print("Precision\t", metrics.box.p)      # overall precision
print("Recall   \t", metrics.box.r)      # overall recall